# [5] Detecting LLM-Generated Text with Binoculars

:ref: https://huggingface.co/blog/dmicz/binoculars-text-detection

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import SWUnivDaconDataset

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from torch.utils.data import DataLoader
from torch.nn import functional as F
from torch import nn
import torch

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
# Change according to hardware
DEVICE_1 = "cuda:0"
DEVICE_2 = "cpu"

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = SWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = SWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

In [ ]:
torch.set_grad_enabled(False)

observer_name = "tiiuae/falcon-7b-instruct"
performer_name = "tiiuae/falcon-7b"

# Verify tokenizers are identical
observer_tokenizer = AutoTokenizer.from_pretrained(observer_name)
performer_tokenizer = AutoTokenizer.from_pretrained(performer_name)

if observer_tokenizer.vocab != performer_tokenizer.vocab:
    raise ValueError("Observer and performer models must have identical tokenizers")

In [ ]:
torch.set_grad_enabled(False)

observer_name = "google/gemma-3-4b-it"
performer_name = "google/gemma-3-4b-pt"

# Verify tokenizers are identical
observer_tokenizer = AutoTokenizer.from_pretrained(observer_name)
performer_tokenizer = AutoTokenizer.from_pretrained(performer_name)

if observer_tokenizer.vocab != performer_tokenizer.vocab:
    raise ValueError("Observer and performer models must have identical tokenizers")

In [ ]:
# Load models with quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": None,
    "quantization_config": quantization_config
}

In [ ]:
observer_model = AutoModelForCausalLM.from_pretrained(
    observer_name, device_map={"": DEVICE_1}, **model_kwargs
)

performer_model = AutoModelForCausalLM.from_pretrained(
    performer_name, device_map={"": DEVICE_2}, **model_kwargs
)

observer_model.eval()
performer_model.eval()

tokenizer = AutoTokenizer.from_pretrained(observer_name)

In [ ]:
def tokenize(text):
    return tokenizer(text, return_tensors="pt")

In [ ]:
tokenize("Hello, my dog is cute")

## Perplexity and cross-perplexity

In [ ]:
criterion = nn.CrossEntropyLoss(reduction='none')

In [ ]:
@torch.inference_mode()
def get_logits(encodings):
    observer_logits = observer_model(**encodings.to(DEVICE_1)).logits
    performer_logits = performer_model(**encodings.to(DEVICE_2)).logits
    return observer_logits, performer_logits

In [ ]:
encoding = tokenize('''Dr. Capy Cosmos, a capybara unlike any other, astounded the scientific community with his
groundbreaking research in astrophysics. With his keen sense of observation and unparalleled ability to interpret
cosmic data, he uncovered new insights into the mysteries of black holes and the origins of the universe. As he
peered through telescopes with his large, round eyes, fellow researchers often remarked that it seemed as if the
stars themselves whispered their secrets directly to him. Dr. Cosmos not only became a beacon of inspiration to
aspiring scientists but also proved that intellect and innovation can be found in the most unexpected of creatures.''')
encoding

In [ ]:
observer_logits, performer_logits = get_logits(encoding)
observer_logits, performer_logits

In [ ]:
encoding.input_ids.shape, observer_logits.shape

In [ ]:
S = observer_logits.shape[-2]
V = observer_logits.shape[-1]

observer_logits[..., :-1, :].contiguous().shape

In [ ]:
encoding.input_ids[..., 1:].shape

In [ ]:
ppl = criterion(observer_logits[..., :-1, :].contiguous().transpose(1, 2).to("cpu"),
                encoding.input_ids[..., 1:].contiguous().to("cpu")).float()

ppl, ppl.sum(1)

In [ ]:
softmax = nn.Softmax(dim=-1)
performer_probs = softmax(performer_logits).view(-1, V)
performer_probs, performer_probs.shape

In [ ]:
observer_scores = observer_logits.view(-1, V).to("cpu")
observer_scores, observer_scores.shape

In [ ]:
xppl = criterion(observer_scores[:-1], performer_probs[:-1]).view(-1, S - 1)

xppl, xppl.sum(1)

In [ ]:
binocular_score = ppl.sum(1) / xppl.sum(1)

binocular_score

In [ ]:
# redefine to handle batch of strings
def tokenize(batch):
    return tokenizer(
        batch, return_tensors="pt", return_token_type_ids=False,
        padding="longest" if len(batch) > 1 else False
    ).to(DEVICE_1)

# redefinition with cuda sync
def get_logits(encodings):
    observer_logits = observer_model(**encodings.to(DEVICE_1)).logits
    performer_logits = performer_model(**encodings.to(DEVICE_2)).logits
    torch.cuda.synchronize()

    return observer_logits, performer_logits

In [ ]:
def perplexity(encoding, logits):
    shifted_logits = logits[..., :-1, :].contiguous()
    shifted_labels = encoding.input_ids[..., 1:].contiguous()
    shifted_attention_mask = encoding.attention_mask[..., 1:].contiguous()

    ppl = criterion(shifted_logits.transpose(1, 2).to("cpu"), shifted_labels) * shifted_attention_mask
    ppl = ppl.sum(1) / shifted_attention_mask.sum(1)

    return ppl.to("cpu").float().numpy()

In [ ]:
def cross_perplexity(observer_logits, performer_logits, encoding):
    V = observer_logits.shape[-1]
    S = observer_logits.shape[-2]

    performer_probs = softmax(performer_logits).view(-1, V).to("cpu")
    observer_scores = observer_logits.view(-1, V).to("cpu")

    xppl = criterion(observer_scores, performer_probs).view(-1, S)
    padding_mask = (encoding.input_ids != tokenizer.pad_token_id).type(torch.uint8)

    xppl = (xppl * padding_mask).sum(1) / padding_mask.sum(1)

    return xppl.to("cpu").float().numpy()

In [ ]:
def binocular_score(text):
    batch = [text] if isinstance(text, str) else text
    encodings = tokenize(batch)
    observer_logits, performer_logits = get_logits(encodings)
    ppl = perplexity(encodings, observer_logits)
    xppl = cross_perplexity(observer_logits, performer_logits, encodings)

    return (ppl / xppl).tolist()

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
tests = ['''The motivation behind LLM Detection is harm reduction, to trace text origins, block spam, and identify fake news produced by LLMs. *Preemptive detection* methods attempt to "watermark" generated text, but requires full control of the generating models, which already seems to be impossible. Therefore, more recent works have been on *post-hoc detection* methods, which could be used without the cooperation of the text's author. The paper's authors suggest that there are two main groups for post-hoc detectors, the first being finetuning a pretrained language model to perform binary classification. There are many additional techniques that make this approach more effective, but all implementations will require training on text produced by the target model, which is both computationally expensive and limited by the number of new models that are being open-sourced.
The second group uses statistical signatures of machine-generated text, with the aim of zero-shot learning. This would allow for the detection of a wide range of models, with little to no training data. These methods use measures such as perplexity, perplexity curvature, log rank, intrinsic dimensionality, and n-gram analysis. The Binoculars paper proposes a focus on low false positive rate (FPR) and high performance on out-of-domain samples, rather than focusing on classifier AUCs for the high-stakes application of LLM detection.''',
 '''Dr. Capy Cosmos, a capybara unlike any other, astounded the scientific community with his
 groundbreaking research in astrophysics. With his keen sense of observation and unparalleled ability to interpret
 cosmic data, he uncovered new insights into the mysteries of black holes and the origins of the universe. As he
 peered through telescopes with his large, round eyes, fellow researchers often remarked that it seemed as if the
 stars themselves whispered their secrets directly to him. Dr. Cosmos not only became a beacon of inspiration to
 aspiring scientists but also proved that intellect and innovation can be found in the most unexpected of creatures.''',
 '''We the People of the United States, in Order to form a more perfect Union, establish Justice, insure domestic Tranquility, provide for the common defence, promote the general Welfare, and secure the Blessings of Liberty to ourselves and our Posterity, do ordain and establish this Constitution for the United States of America.'''
 ]
binocular_score(tests)